<a href="https://colab.research.google.com/github/ravindyaparami/multi-agents-on-a-lie-group/blob/main/Drake/Rigid_Body.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install drake


In [ ]:
from pydrake.all import *
import numpy as np

In [ ]:
builder = DiagramBuilder()


In [ ]:
plant, scene_graph = AddMultibodyPlantSceneGraph(
    builder,
    MultibodyPlant(time_step=0.001)
)


In [ ]:
mass = 1.0  # kg
box_size = [0.2, 0.2, 0.2]  # meters

inertia = UnitInertia.SolidBox(*box_size)
spatial_inertia = SpatialInertia(
    mass=mass,
    p_PScm_E=np.zeros(3),
    G_SP_E=inertia
)

body = plant.AddRigidBody(
    "box",
    spatial_inertia
)


In [ ]:
plant.SetDefaultFloatingBaseBodyPose(
    body,
    RigidTransform([0, 0, 1.0])
)



In [ ]:
box_shape = Box(*box_size)

plant.RegisterVisualGeometry(
    body,
    RigidTransform(),
    box_shape,
    "box_visual",
    [0.5, 0.5, 1.0, 1.0]  # RGBA
)


<GeometryId value=18>

In [ ]:
plant.mutable_gravity_field().set_gravity_vector([0, 0, -9.81])


In [ ]:
ground_shape = HalfSpace()
X_WG = RigidTransform(RollPitchYaw(np.pi, 0, 0),
                      [0, 0, 0] )

In [ ]:
plant.RegisterCollisionGeometry(
    plant.world_body(),
    X_WG,
    ground_shape,
    "ground_collision",
    CoulombFriction(0.9, 0.8)
)

<GeometryId value=20>

In [ ]:
plant.RegisterVisualGeometry(
    plant.world_body(),
    X_WG,
    ground_shape,
    "ground_visual",
    [0.5, 0.5, 0.5, 1.0]
)

<GeometryId value=22>

In [ ]:
plant.Finalize()


In [ ]:
meshcat = StartMeshcat()

visualizer = MeshcatVisualizer.AddToBuilder(
    builder,
    scene_graph,
    meshcat
)

'''from IPython.display import HTML
display(HTML(f'<a href="{meshcat.web_url()}" target="_blank">Open Meshcat Viewer</a>'))
'''

INFO:drake:Meshcat listening for connections at http://localhost:7000


'from IPython.display import HTML\ndisplay(HTML(f\'<a href="{meshcat.web_url()}" target="_blank">Open Meshcat Viewer</a>\'))\n'

In [ ]:
diagram = builder.Build()


In [ ]:
simulator = Simulator(diagram)
simulator.set_target_realtime_rate(1.0)

context = simulator.get_mutable_context()


In [ ]:
simulator.Initialize()
simulator.AdvanceTo(5.0)
